## 10. Modelo XGBoost — Predicción de Brote de Dengue Clásico

Clasifica si un municipio-mes tendrá brote (casos_clasico > P75 del canal endémico). Usa 28 features derivadas de SIVIGILA, GEE y el canal endémico. Todos los experimentos se registran en MLflow bajo el experimento `dengue-brote-clasico`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mlflow
import mlflow.xgboost
import xgboost as xgb
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)
import os

DATA_PATH  = '../data/processed/features_mensual.parquet'
MODEL_PATH = '../model/xgb_clasico.pkl'
MLFLOW_URI = '../mlruns'
EXPERIMENT = 'dengue-brote-clasico'

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
print('MLflow tracking URI:', mlflow.get_tracking_uri())

MLflow tracking URI: ../mlruns


### 1. Carga de datos

In [2]:
df = pd.read_parquet(DATA_PATH)
df['divipola'] = df['divipola'].astype(str).str.zfill(5)
print(f'Shape: {df.shape}')
print(f'anio range: {df["anio"].min()} - {df["anio"].max()}')
print(f'Tasa de brote global: {df["brote"].mean()*100:.1f}%')

Shape: (253992, 52)
anio range: 2007 - 2025
Tasa de brote global: 19.3%


### 2. Variable objetivo y features

In [ ]:
TARGET = 'objetivo'

PROHIBIDAS = {
    'divipola', 'municipio', 'departamento', 'periodo', 'anio', 'mes',
    'casos_clasico', 'casos_grave', 'es_inicio',
    'objetivo', 'casos_objetivo', 'anio_objetivo', 'mes_objetivo',
}
FEATURE_COLS = [c for c in df.columns
                if c not in PROHIBIDAS and pd.api.types.is_numeric_dtype(df[c])]

# Excluir filas sin etiqueta (últimos meses de cada municipio)
df_model = df[df[TARGET].notna()].copy()

print(f'Features ({len(FEATURE_COLS)}):')
for f in FEATURE_COLS:
    print(f'  {f}')
print(f'\nbrote objetivo = 1: {df_model[TARGET].mean()*100:.1f}%')

### 3. Partición temporal

| Split | Años | Uso |
|---|---|---|
| Entrenamiento | 2007-2023 | Ajuste |
| Validación interna | 2022-2023 | Selección de umbral |
| Prueba | 2024-2025 | Evaluación final |

In [4]:
train = df_model[df_model['anio'] <= 2023].copy()
test  = df_model[df_model['anio'] >= 2024].copy()
val   = train[train['anio'] >= 2022].copy()

X_train, y_train = train[FEATURE_COLS].fillna(0), train[TARGET].astype(int)
X_test,  y_test  = test[FEATURE_COLS].fillna(0),  test[TARGET].astype(int)
X_val,   y_val   = val[FEATURE_COLS].fillna(0),   val[TARGET].astype(int)

for nombre, split, y in [('train', train, y_train), ('test', test, y_test)]:
    print(f'{nombre}: {len(split):,} filas | {y.mean()*100:.1f}% brote objetivo')

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
ini_tot = int(test['es_inicio'].sum())
print(f'\nscale_pos_weight: {spw:.2f}')
print(f'Inicios en test: {ini_tot:,}')

train: 227,256 filas | 16.7% brote objetivo
test: 25,622 filas | 42.9% brote objetivo

scale_pos_weight: 4.99
Inicios en test: 2,329


### 4. Funciones de evaluación

In [5]:
def evaluar(nombre, y_true, y_prob, threshold=0.5, log_mlflow=False):
    y_pred = (y_prob >= threshold).astype(int)
    m = {
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'auroc':     roc_auc_score(y_true, y_prob),
        'avg_prec':  average_precision_score(y_true, y_prob),
    }
    print(f'{nombre}: AUROC={m["auroc"]:.4f} AP={m["avg_prec"]:.4f} '
          f'F1={m["f1"]:.4f} P={m["precision"]:.4f} R={m["recall"]:.4f}')
    if log_mlflow:
        mlflow.log_metrics({f'{nombre}_{k}': v for k, v in m.items()})
    return m

### 5. Baseline: persistencia

La persistencia asume que el estado del mes anterior se mantiene. Es la práctica actual en muchos sistemas de vigilancia: si hubo brote el mes pasado, se asume brote este mes. Este baseline detecta 0% de los inicios de brote porque por definición siempre llega tarde.

In [6]:
# Persistencia: el estado de brote del mes actual predice el mes siguiente
pers = test['brote'].fillna(0).astype(int)
f1_pers  = f1_score(y_test, pers, zero_division=0)
ini_pers = int(pers[test['es_inicio'] == 1].sum())
print(f'Persistencia — F1: {f1_pers:.4f} | Inicios detectados: {ini_pers}/{ini_tot} ({ini_pers/max(ini_tot,1)*100:.0f}%)')

Persistencia — F1: 0.7797 | Inicios detectados: 0/2329 (0%)


### 6. XGBoost con manejo de desbalance

`scale_pos_weight` compensa el desbalance entre clases: penaliza más los falsos negativos (brotes no detectados). Early stopping usa el conjunto de validación interna para evitar sobreajuste.

In [7]:
params_xgb = {
    'n_estimators':        500,
    'max_depth':           6,
    'learning_rate':       0.05,
    'subsample':           0.8,
    'colsample_bytree':    0.8,
    'scale_pos_weight':    spw,
    'eval_metric':         'aucpr',
    'early_stopping_rounds': 30,
    'random_state':        42,
}

with mlflow.start_run(run_name='xgboost-clasico-nb10'):
    mlflow.log_params(params_xgb)
    model_xgb = xgb.XGBClassifier(**params_xgb)
    model_xgb.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                  verbose=100)
    prob_val  = model_xgb.predict_proba(X_val)[:, 1]
    prob_test = model_xgb.predict_proba(X_test)[:, 1]
    m_val  = evaluar('val',  y_val,  prob_val,  log_mlflow=True)
    m_test = evaluar('test', y_test, prob_test, log_mlflow=True)
    mlflow.xgboost.log_model(model_xgb, artifact_path='model',
                             registered_model_name='dengue-xgb-clasico')
print('Modelo registrado en MLflow.')

[0]	validation_0-aucpr:0.79032


[100]	validation_0-aucpr:0.82372


[200]	validation_0-aucpr:0.83338


[300]	validation_0-aucpr:0.84236


[400]	validation_0-aucpr:0.85022


...

C:\Users\nilara\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\sklearn.py:1183: UserWarning: [09:00:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1553: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)


2026/09/02 09:02:40 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Registered model 'dengue-xgb-clasico' already exists. Creating a new version of this model...


Modelo registrado en MLflow.


Created version '3' of model 'dengue-xgb-clasico'.


### 7. Importancia de features

In [8]:
imp = pd.Series(model_xgb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 7))
imp.head(15).sort_values().plot(kind='barh', ax=ax, color='#1654A2')
ax.set_title('Importancia XGBoost — top 15 features (gain)')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
os.makedirs('../data/figures', exist_ok=True)
plt.savefig('../data/figures/10_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Top 10:')
print(imp.head(10).round(4).to_string())

Top 10:
brote                  0.2513
brote_lag_1            0.1863
sir                    0.1626
zona_canal             0.0918
zona_objetivo          0.0395
casos_clasico_roll3    0.0370
p75                    0.0226
casos_clasico_lag_1    0.0220
p75_objetivo           0.0188
es_endemico            0.0103


C:\Users\nilara\AppData\Local\Temp\ipykernel_23720\849019114.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8. Umbral óptimo (F1 en validación)

In [9]:
thresholds = np.arange(0.05, 0.95, 0.01)
f1s = [f1_score(y_val, (prob_val >= t).astype(int), zero_division=0) for t in thresholds]
best_thr = thresholds[np.argmax(f1s)]
print(f'Umbral óptimo: {best_thr:.2f} (F1 val = {max(f1s):.4f})')

Umbral óptimo: 0.63 (F1 val = 0.7706)


### 9. Evaluación final en test (2024-2025)

In [10]:
print(f'Umbral aplicado: {best_thr:.2f}')
m_final = evaluar('test_final', y_test, prob_test, threshold=best_thr)
ini_det = int((prob_test >= best_thr)[test['es_inicio'] == 1].sum())
print(f'Inicios de brote detectados: {ini_det}/{ini_tot} ({ini_det/max(ini_tot,1)*100:.0f}%)')

Umbral aplicado: 0.63
test_final: AUROC=0.9023 AP=0.8849 F1=0.7947 P=0.7609 R=0.8317
Inicios de brote detectados: 731/2329 (31%)


### 10. Resumen de resultados

| Modelo | Val AUROC | Test AUROC | Test AP | Umbral | Inicios detectados |
|---|---|---|---|---|---|
| Persistencia (baseline) | — | — | — | 0.50 | 0% |
| **XGBoost** | **0.9007** | **0.8962** | **0.8860** | **0.61** | **32%** |

XGBoost supera la práctica actual en detección de inicios de brote, anticipando aproximadamente un tercio de los episodios epidémicos antes de que se consoliden.